# EDA

In [13]:
import re
from pathlib import Path
from typing import Literal

import altair as alt
import pandas as pd
import pymupdf

type Partition = Literal["train", "dev-0", "test-A"]


outputs_path = Path().cwd() / "src" / "nda" / "static" / "outputs"

## Import data

### Read parquet files

In [2]:
def read_parquet_file(partition: Partition) -> pd.DataFrame:
    file_path = outputs_path / partition / "data.parquet"

    if not file_path.exists():
        raise FileNotFoundError(f"Parquet file not found at {file_path}")

    if partition == "test-A":
        return pd.read_parquet(file_path).loc[
            :,
            [
                "filename",
                "text_best",
            ],
        ]
    return pd.read_parquet(file_path).loc[
        :,
        [
            "filename",
            "labels",
            "labels_schema",
            "text_best",
        ],
    ]

In [3]:
train, val, test = [
    read_parquet_file(partition) for partition in ("train", "dev-0", "test-A")
]

train.head()

,filename,labels,labels_schema,text_best
0,00a1d238e37ac225b8045a97953e845d.pdf,effective_date=2001-04-18 jurisdiction=Oregon ...,"{'effective_date': '2001-04-18', 'jurisdiction...",EX-10.23 5 dex1023.htm COVENANT NOT TO COMPETE...
1,031470434423a8c40105a4b404ced88b.pdf,effective_date=2017-02-10 jurisdiction=Califor...,"{'effective_date': '2017-02-10', 'jurisdiction...",EX-99.(E)(2) 3 d450961dex99e2.htm EX-(E)(2)\nE...
2,03ae3b511276b560dc8806eb61b9d063.pdf,effective_date=2012-01-06 jurisdiction=Florida...,"{'effective_date': '2012-01-06', 'jurisdiction...",EX-10.3 6 d281487dex103.htm CONFIDENTIALITY AN...
3,03efbda01358533c167ca9b1e6d72051.pdf,effective_date=1999-02-08 jurisdiction=Pennsyl...,"{'effective_date': '1999-02-08', 'jurisdiction...",EX-10.26 26 ex10-26.txt NON-CIRCUMVENTION AND ...
4,03fd0e629b617da00c54794a8a78b24d.pdf,effective_date=2011-07-13 jurisdiction=Califor...,"{'effective_date': '2011-07-13', 'jurisdiction...",EX-7.5 2 dex75.htm AMENDED AND RESTATED CONFID...


### Enrich with document metadata

In [4]:
def get_document_metadata(documents_path: Path, filename: str) -> dict:
    with pymupdf.open(documents_path / filename) as doc:
        page_count = doc.page_count
        full_text = "\n".join(str(page.get_text("text")) for page in doc)
        full_text = re.sub(r"\xa0+", " ", full_text)
        full_text = re.sub(r"\n \n", "\n", full_text)

    return {
        "filename": filename,
        "text_pymupdf": full_text,
        "char_count": len(full_text),
        "word_count": len(re.split(r"\s+", full_text.strip())),
        "line_count": full_text.count("\n") + 1,
        "page_count": page_count,
    }

In [5]:
def enrich_with_document_metadata(df: pd.DataFrame, partition: Partition) -> pd.DataFrame:
    documents_path = outputs_path / partition / "documents"
    metadata = [get_document_metadata(documents_path, filename) for filename in df["filename"]]
    return df.merge(pd.DataFrame(metadata), on="filename", how="left")

In [6]:
train, val, test = [
    enrich_with_document_metadata(df, partition) 
    for df, partition in zip((train, val, test), ("train", "dev-0", "test-A"), strict=True)
]

train.head()

,filename,labels,labels_schema,text_best,text_pymupdf,char_count,word_count,line_count,page_count
0,00a1d238e37ac225b8045a97953e845d.pdf,effective_date=2001-04-18 jurisdiction=Oregon ...,"{'effective_date': '2001-04-18', 'jurisdiction...",EX-10.23 5 dex1023.htm COVENANT NOT TO COMPETE...,EX-10.23 5 dex1023.htm COVENANT NOT TO COMPETE...,11125,1699,121,5
1,031470434423a8c40105a4b404ced88b.pdf,effective_date=2017-02-10 jurisdiction=Califor...,"{'effective_date': '2017-02-10', 'jurisdiction...",EX-99.(E)(2) 3 d450961dex99e2.htm EX-(E)(2)\nE...,EX-99.(E)(2) 3 d450961dex99e2.htm EX-(E)(2)\nE...,16104,2409,241,5
2,03ae3b511276b560dc8806eb61b9d063.pdf,effective_date=2012-01-06 jurisdiction=Florida...,"{'effective_date': '2012-01-06', 'jurisdiction...",EX-10.3 6 d281487dex103.htm CONFIDENTIALITY AN...,EX-10.3 6 d281487dex103.htm CONFIDENTIALITY AN...,34535,5168,388,10
3,03efbda01358533c167ca9b1e6d72051.pdf,effective_date=1999-02-08 jurisdiction=Pennsyl...,"{'effective_date': '1999-02-08', 'jurisdiction...",EX-10.26 26 ex10-26.txt NON-CIRCUMVENTION AND ...,EX-10.26 26 ex10-26.txt NON-CIRCUMVENTION AND ...,7474,1144,66,2
4,03fd0e629b617da00c54794a8a78b24d.pdf,effective_date=2011-07-13 jurisdiction=Califor...,"{'effective_date': '2011-07-13', 'jurisdiction...",EX-7.5 2 dex75.htm AMENDED AND RESTATED CONFID...,EX-7.5 2 dex75.htm AMENDED AND RESTATED CONFID...,24766,3838,244,9


## Descriptive statistics

In [24]:
def fetch_statistics(df: pd.DataFrame) -> pd.DataFrame:
    statistics = ["mean", "median", "min", "max", "kurtosis", "skew"]

    return df.aggregate({
        "char_count": statistics,
        "word_count": statistics,
        "line_count": statistics,
        "page_count": statistics,
    })


def check_distributions(df: pd.DataFrame, partition: Partition) -> alt.HConcatChart:
    metrics = ["char_count", "word_count", "line_count", "page_count"]

    charts = [
        alt.Chart(df[[metric]]).mark_bar().encode(
            x=alt.X(f"{metric}:Q", bin=alt.Bin(maxbins=30), title=metric),
            y=alt.Y("count()", title="count"),
        ).properties(title=metric, width=200, height=200)
        for metric in metrics
    ]

    return alt.hconcat(*charts).properties(title=f"Document metric distributions ({partition})")

### Train

In [8]:
fetch_statistics(train)

,char_count,word_count,line_count,page_count
mean,19399.755906,2993.881890,208.055118,6.440945
median,17104.000000,2670.500000,185.500000,6.000000
min,3258.000000,493.000000,25.000000,1.000000
max,72385.000000,11191.000000,712.000000,23.000000
kurtosis,5.242843,5.021513,3.755031,4.176024
skew,1.702093,1.662945,1.581395,1.646698


In [25]:
check_distributions(train, "train")

alt.HConcatChart(...)

### Validation

In [9]:
fetch_statistics(val)

,char_count,word_count,line_count,page_count
mean,16982.108434,2599.385542,184.807229,5.891566
median,15550.000000,2387.000000,179.000000,5.000000
min,1799.000000,249.000000,26.000000,1.000000
max,44097.000000,6612.000000,493.000000,17.000000
kurtosis,0.622117,0.409938,1.611105,1.861946
skew,0.711850,0.687246,0.990803,1.245817


In [27]:
check_distributions(val, "dev-0")

alt.HConcatChart(...)

### Test

In [10]:
fetch_statistics(test)

,char_count,word_count,line_count,page_count
mean,15835.463054,2423.995074,178.226601,5.428571
median,13683.000000,2114.000000,161.000000,5.000000
min,2849.000000,443.000000,40.000000,1.000000
max,58081.000000,9009.000000,762.000000,23.000000
kurtosis,4.703208,4.988329,7.592340,4.126699
skew,1.788207,1.838904,2.158560,1.698986


In [28]:
check_distributions(test, "test-A")

alt.HConcatChart(...)

## Label distribution

## Document generalities